In [2]:
from langchain_ollama import OllamaLLM
import pandas as pd
import numpy as np
import re
from cleantext import clean
Words2Rmv = ['xxx']

In [3]:
llm = OllamaLLM(model="financial_analyser_llama2:latest")
llm.invoke("Who is leading using AI to fight financial crime?")

"\nAs a financial assistant, I can tell you that several organizations and companies are leveraging artificial intelligence (AI) to combat financial crime. Here are some examples:\n\n1. FinCEN: The Financial Crimes Enforcement Network (FinCEN) is a bureau of the United States Department of the Treasury that is responsible for implementing and enforcing the Bank Secrecy Act (BSA). FinCEN uses AI and machine learning to analyze large volumes of data to identify potential money laundering and terrorist financing activities.\n2. Interpol: Interpol is an international law enforcement agency that is working to combat financial crime through its Financial Crime Programme. Interpol uses AI and other advanced technologies to analyze financial transactions and identify potential criminal activity.\n3. SWIFT: SWIFT is a global financial messaging service that is used by thousands of banks and other financial institutions worldwide. SWIFT has implemented an AI-powered system called KYC (Know Your 

In [4]:
def read_data():
    TransDF = pd.read_csv('sample100.csv')
    TransDF['Amount'] = pd.to_numeric(TransDF['Amount'], errors='coerce').fillna(0).astype(int)
    
    return TransDF

AllTransDF = read_data()

#make sure the data has the right data type
print(AllTransDF.dtypes)

Date             str
Amount         int64
Transaction      str
dtype: object


In [5]:
AllTransDF.head()

,Date,Amount,Transaction
0,30/07/2025,-4,BACI PASTA CAFE
1,30/12/2025,-9,KMART
2,07/01/2026,-197,BARWON WATER
3,15/01/2025,-62,TONG LI SUPERMARKETS
4,12/11/2025,-3895,AGODA.COM ALUNA HOTEL


In [6]:
def transform_data(AllTransDF):
    
    pd.set_option('display.max_colwidth', None)
    #remove unwanted words
    AllTransDF['CleanTransaction'] = AllTransDF['Transaction'].apply(
        lambda x: ' '.join([word for word in x.split() 
                            if word.lower() not in Words2Rmv])
    )
    
    #perform some simple text cleaning using cleantext
    AllTransDF['Transaction'] = AllTransDF['CleanTransaction'].apply(lambda x: clean(
        x,
        clean_all=True,
        extra_spaces=True,
        numbers=True,
        punct=True
        ))
    
    AllTransDF = AllTransDF.drop("CleanTransaction", axis=1)
    AllTransDF['Income/Expense'] = np.where(AllTransDF['Amount']>0, 'Income','Expense')
    AllTransDF['Transaction']=AllTransDF['Transaction'].str.title()
   
    return (AllTransDF)

TransformedDF = transform_data(AllTransDF)

In [7]:
UniqueTransactions=TransformedDF['Transaction'].unique()
len(UniqueTransactions)
UniqueTransactions

<StringArray>
[                     'Baci Pasta Cafe',
                                'Kmart',
                         'Barwon Water',
                 'Tong Li Supermarkets',
                 'Agodacom Aluna Hotel',
                'Aust Hlth Mgmt Grp Pl',
                           'Opal Topup',
                    'Transfer From Cba',
                          'Aldi Stores',
       'Transfer From Metro Art Galary',
                            'Big Apple',
              'Woolworthsgreat Western',
                                'Coles',
                               'Virgin',
      'Payment To Occomptyltd Internet',
               'Payment To Credit Card',
                            'Sushi Hub',
                     'Payment From Cba',
                         'Aaa Artcraft',
 'Woolworthsstrathfield Pl Strathfield',
                                 'Myer',
                    'Les Family Clinic',
                         'Loan Payment',
                         'Payment From',
  

In [8]:
response = llm.invoke('can you add a proper category to the following items. Do not apply reasonong. ' \
'Responde with category. For example: Coles Express Bread - groceries, Big Apple transaction - groceries' + 
', '.join(UniqueTransactions[1:10]))
response = response.split('\n')
response

['',
 'Sure! Here are the items you provided and their corresponding categories:',
 '',
 '1. Coles Express Bread - Groceries',
 '2. Big Apple transaction - Groceries',
 '3. Kmart - Retail',
 '4. Barwon Water - Utilities',
 '5. Tong Li Supermarkets - Groceries',
 '6. Agodacom Aluna Hotel - Travel',
 '7. Aust Hlth Mgmt Grp Pl - Insurance',
 '8. Opal Topup - Transportation',
 '9. Transfer From Cba - Financial Services',
 '10. Aldi Stores - Retail',
 '11. Transfer From Metro Art Galary - Entertainment']

In [9]:
# Generate an arrary of sequence of 10s and append the final boundary index
IndexTransList = (
    np.append(np.arange(0, len(UniqueTransactions), 10), 
              len(UniqueTransactions)).tolist()
)

IndexTransList


[0, 10, 20, 30, 40, 50, 60, 65]

In [10]:
# Validate response 's format
from pydantic import BaseModel, field_validator
from typing import List

class ResponseChecks(BaseModel):
    data: List[str]
    @field_validator('data')
    def check_length(cls, value):
        for item in value:
            if len(item) >0:
                assert "-" in item, f"Item '{item}' does not contain a hyphen"
#Pass asertion
ResponseChecks(data=['Hello World - greeting', 'banana cake - fruit'])
                    
    

ResponseChecks(data=None)

In [19]:
#if the check will fail
ResponseChecks(data=['Hello-World', 'banana fruit'])

ValidationError: 1 validation error for ResponseChecks
data
  Assertion failed, Item 'banana fruit' does not contain a hyphen [type=assertion_error, input_value=['Hello-World', 'banana fruit'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/assertion_error

In [14]:
def categorise_transactions(transactions, llm):
    response = llm.invoke("Can you add an appropriate category to the following expenses. For example: " \
    "Best Baking Syndey Metro - food, oil paint canvas - creative, uberdrive int aiport - transport, utility Sept - utility. " \
    "Categories should be less than 4 characters. " + transactions)
    response = response.split('\n')
    
    #removing the explaination lines at the beginning and end of the response
    TransIndexes = [index for index in range(len(response)) if response[index] == '']
    if len(TransIndexes) == 1:
        response = response[(TransIndexes[0] + 1):]
    else:
        response = response[(TransIndexes[0] + 1) : TransIndexes[1]]

    #validate response for '-' to separate transaction and category
    ResponseChecks(data = response)
    
    # assigned the response to a dataframe
    CategoriesDF = pd.DataFrame({'Transaction - Category': response})
    CategoriesDF[['Transaction', 'Category']] = CategoriesDF['Transaction - Category'].str.split(' - ', expand=True)

    return CategoriesDF


In [15]:
#test the categorise_transactions function
CatTrans = categorise_transactions('Best Baking Syndey Metro, oil paint canvas , uberdrive int aiport, utility Sept',llm)
CatTrans

,Transaction - Category,Transaction,Category
0,1. Best Baking Syndey Metro - food,1. Best Baking Syndey Metro,food
1,2. Oil paint canvas - creative,2. Oil paint canvas,creative
2,3. Uber drive Int Airport - transport,3. Uber drive Int Airport,transport
3,4. Utility Sept - utility,4. Utility Sept,utility


In [16]:
def process_categories(UniqueTransactions, IndexTransList, llm):
  
    #initialise a new empty dataframe
    CategoriesDF = pd.DataFrame()
    #set a max tries if LLM output incorrect format. Rule set '-' between transaction and category
    MaxTries = 5

    #loop through the unique transactions and get the block of categories using index list built before
    for i in range(0,len(IndexTransList)-1):
        #extract transaction belong to current block
        BlockTrans= UniqueTransactions[IndexTransList[i]:IndexTransList[i+1]]
        #Separate each transaction with a comma before fit them to LLM
        BlockTrans = ', '.join(BlockTrans)
        # retry loop when LLM return output of incorrect format
        for j in range(MaxTries):
            try:
                #call llm to classify the block of transactions
                Categories = categorise_transactions(BlockTrans, llm)
                #Append the output to the dataframe.
                CategoriesDF = pd.concat([CategoriesDF, Categories], ignore_index=True)                                 
            except:
                if j < MaxTries:
                    continue
                else:
                    raise Exception (f"Failed to categorise transactions after {MaxTries} attempts.")
            break
    return CategoriesDF
#process all uniques categories
CategoriesDF = process_categories(UniqueTransactions, IndexTransList, llm)
CategoriesDF

,Transaction - Category,Transaction,Category
0,1. Baci Pasta Cafe - Food,1. Baci Pasta Cafe,Food
1,2. Kmart - Shopping,2. Kmart,Shopping
2,3. Barwon Water - Utilities,3. Barwon Water,Utilities
3,4. Tong Li Supermarkets - Groceries,4. Tong Li Supermarkets,Groceries
4,5. Agodacom Aluna Hotel - Accommodation,5. Agodacom Aluna Hotel,Accommodation
5,6. Aust Hlth Mgmt Grp Pl - Unknown,6. Aust Hlth Mgmt Grp Pl,Unknown
6,7. Opal Topup - Transport,7. Opal Topup,Transport
7,8. Transfer From Cba - Unknown,8. Transfer From Cba,Unknown
8,9. Aldi Stores - Groceries,9. Aldi Stores,Groceries
9,10. Transfer From Metro Art Galary - Unknown,10. Transfer From Metro Art Galary,Unknown


In [17]:
#remove unwanted characters leading each transaction
CategoriesDF['Transaction'] = CategoriesDF['Transaction'].str.replace(r'^(\d+\.|\*)\s+', '', regex=True).str.strip()
CategoriesDF['Category']=CategoriesDF['Category'].str.title()
CategoriesDF


,Transaction - Category,Transaction,Category
0,1. Baci Pasta Cafe - Food,Baci Pasta Cafe,Food
1,2. Kmart - Shopping,Kmart,Shopping
2,3. Barwon Water - Utilities,Barwon Water,Utilities
3,4. Tong Li Supermarkets - Groceries,Tong Li Supermarkets,Groceries
4,5. Agodacom Aluna Hotel - Accommodation,Agodacom Aluna Hotel,Accommodation
5,6. Aust Hlth Mgmt Grp Pl - Unknown,Aust Hlth Mgmt Grp Pl,Unknown
6,7. Opal Topup - Transport,Opal Topup,Transport
7,8. Transfer From Cba - Unknown,Transfer From Cba,Unknown
8,9. Aldi Stores - Groceries,Aldi Stores,Groceries
9,10. Transfer From Metro Art Galary - Unknown,Transfer From Metro Art Galary,Unknown


In [18]:
SourceDF=TransformedDF
SourceDF = SourceDF.rename(columns={"Transaction": "SrcTransaction"})
CategorisedTrans = pd.merge(
    SourceDF, CategoriesDF, left_on="SrcTransaction", right_on="Transaction", how="left"
)
CategorisedTrans


,Date,Amount,SrcTransaction,Income/Expense,Transaction - Category,Transaction,Category
0,30/07/2025,-4,Baci Pasta Cafe,Expense,1. Baci Pasta Cafe - Food,Baci Pasta Cafe,Food
1,30/12/2025,-9,Kmart,Expense,2. Kmart - Shopping,Kmart,Shopping
2,07/01/2026,-197,Barwon Water,Expense,3. Barwon Water - Utilities,Barwon Water,Utilities
3,15/01/2025,-62,Tong Li Supermarkets,Expense,4. Tong Li Supermarkets - Groceries,Tong Li Supermarkets,Groceries
4,12/11/2025,-3895,Agodacom Aluna Hotel,Expense,5. Agodacom Aluna Hotel - Accommodation,Agodacom Aluna Hotel,Accommodation
...,...,...,...,...,...,...,...
93,19/01/2026,-16,Woolworthsgreat Western,Expense,NaN,NaN,NaN
94,25/05/2026,-19,Woolworthswestfield Camp,Expense,NaN,NaN,NaN
95,18/11/2025,-992,Paid To Credit Card,Expense,NaN,NaN,NaN
96,11/11/2024,167,Sold Osis Painting,Income,NaN,NaN,NaN


In [19]:
CategorisedTrans = CategorisedTrans.dropna()
CategorisedTrans

,Date,Amount,SrcTransaction,Income/Expense,Transaction - Category,Transaction,Category
0,30/07/2025,-4,Baci Pasta Cafe,Expense,1. Baci Pasta Cafe - Food,Baci Pasta Cafe,Food
1,30/12/2025,-9,Kmart,Expense,2. Kmart - Shopping,Kmart,Shopping
2,07/01/2026,-197,Barwon Water,Expense,3. Barwon Water - Utilities,Barwon Water,Utilities
3,15/01/2025,-62,Tong Li Supermarkets,Expense,4. Tong Li Supermarkets - Groceries,Tong Li Supermarkets,Groceries
4,12/11/2025,-3895,Agodacom Aluna Hotel,Expense,5. Agodacom Aluna Hotel - Accommodation,Agodacom Aluna Hotel,Accommodation
5,02/08/2024,-170,Aust Hlth Mgmt Grp Pl,Expense,6. Aust Hlth Mgmt Grp Pl - Unknown,Aust Hlth Mgmt Grp Pl,Unknown
6,08/07/2025,-24,Opal Topup,Expense,7. Opal Topup - Transport,Opal Topup,Transport
7,07/10/2024,5073,Transfer From Cba,Income,8. Transfer From Cba - Unknown,Transfer From Cba,Unknown
8,21/01/2026,-23,Aldi Stores,Expense,9. Aldi Stores - Groceries,Aldi Stores,Groceries
9,19/06/2026,2000,Transfer From Metro Art Galary,Income,10. Transfer From Metro Art Galary - Unknown,Transfer From Metro Art Galary,Unknown


In [21]:
UniqCategories = CategorisedTrans['Category'].unique()
UniqCategories

<StringArray>
[              'Food',           'Shopping',          'Utilities',
          'Groceries',      'Accommodation',            'Unknown',
          'Transport',            'Grocery',      'Entertainment',
 'Financial Services',           'Creative',             'Travel',
            'Utility',   'Home Improvement',  'Health & Wellness',
   'Home Maintenance',           'Clothing',      'Personal Care']
Length: 18, dtype: str

In [22]:
def regroup_category():
    # 2. Define a lookup dictionary
    LookupDic = {
        "Food": "Dining & Leisure",
        "Entertainment": "Dining & Leisure",
        "Groceries": "Supplies & Essentials",
        "Grocery": "Supplies & Essentials",
        "Shopping": "Shopping & Retails",
        "Clothing": "Shopping & Retails",
        "Home": "Shopping & Retails",
        "Utilities": "Utility"
    }
    CategorisedTrans['RefineCategory'] = CategorisedTrans['Category'].map(LookupDic).fillna(CategorisedTrans['Category'])
    return CategorisedTrans

CategorisedTrans = regroup_category()
print(CategorisedTrans[['Category', 'RefineCategory']])

              Category         RefineCategory
0                 Food       Dining & Leisure
1             Shopping     Shopping & Retails
2            Utilities                Utility
3            Groceries  Supplies & Essentials
4        Accommodation          Accommodation
5              Unknown                Unknown
6            Transport              Transport
7              Unknown                Unknown
8            Groceries  Supplies & Essentials
9              Unknown                Unknown
10                Food       Dining & Leisure
12             Grocery  Supplies & Essentials
13       Entertainment       Dining & Leisure
14             Unknown                Unknown
16  Financial Services     Financial Services
17                Food       Dining & Leisure
18  Financial Services     Financial Services
19             Grocery  Supplies & Essentials
20            Creative               Creative
26           Groceries  Supplies & Essentials
27           Utilities            

In [23]:
CategorisedTrans=CategorisedTrans.drop(columns = 'Category').rename(columns ={'RefineCategory': 'Category'})
CategorisedTrans

,Date,Amount,SrcTransaction,Income/Expense,Transaction - Category,Transaction,Category
0,30/07/2025,-4,Baci Pasta Cafe,Expense,1. Baci Pasta Cafe - Food,Baci Pasta Cafe,Dining & Leisure
1,30/12/2025,-9,Kmart,Expense,2. Kmart - Shopping,Kmart,Shopping & Retails
2,07/01/2026,-197,Barwon Water,Expense,3. Barwon Water - Utilities,Barwon Water,Utility
3,15/01/2025,-62,Tong Li Supermarkets,Expense,4. Tong Li Supermarkets - Groceries,Tong Li Supermarkets,Supplies & Essentials
4,12/11/2025,-3895,Agodacom Aluna Hotel,Expense,5. Agodacom Aluna Hotel - Accommodation,Agodacom Aluna Hotel,Accommodation
5,02/08/2024,-170,Aust Hlth Mgmt Grp Pl,Expense,6. Aust Hlth Mgmt Grp Pl - Unknown,Aust Hlth Mgmt Grp Pl,Unknown
6,08/07/2025,-24,Opal Topup,Expense,7. Opal Topup - Transport,Opal Topup,Transport
7,07/10/2024,5073,Transfer From Cba,Income,8. Transfer From Cba - Unknown,Transfer From Cba,Unknown
8,21/01/2026,-23,Aldi Stores,Expense,9. Aldi Stores - Groceries,Aldi Stores,Supplies & Essentials
9,19/06/2026,2000,Transfer From Metro Art Galary,Income,10. Transfer From Metro Art Galary - Unknown,Transfer From Metro Art Galary,Unknown


In [39]:
CategorisedTrans.to_csv('Categorised_cleaned_trans.csv')